# Confusion matrix & classification metrics

_Machine Learning_

**Count the right and wrong predictions, then score the classifier.**

Accuracy alone can lie, especially when one class is rare.

     A confusion matrix counts four outcomes: true/false positives and negatives.

---

This notebook builds up a confusion matrix and the metrics that come from it, one step at a time, on a deliberately imbalanced dataset. Run each cell, read the note above it, and watch how the metrics tell a richer story than accuracy alone. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We train a classifier on a deliberately **imbalanced** dataset (70% one class, 30% the other) and read off the confusion matrix plus the full metric report. We build it in three steps: (1) make an imbalanced dataset and split it, (2) fit a logistic regression, (3) score it with a confusion matrix and per-class precision/recall/F1.

### Step 1 — Make an imbalanced dataset and split it

We generate a 2-class problem where the classes are uneven (`weights=[0.7, 0.3]`). Imbalance is exactly the situation where accuracy can mislead: a model that always predicts the majority class scores 70% without learning anything. We hold out 30% as a test set to score honestly.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# 500 examples, 6 features, classes split 70% / 30% (imbalanced on purpose).
X, y = make_classification(n_samples=500, n_features=6, weights=[0.7, 0.3],
                           random_state=0)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

### Step 2 — Fit the classifier and predict

We fit a plain `LogisticRegression` on the training split and ask it for hard class predictions on the held-out test set. These predictions are what we compare against the true labels to count right and wrong outcomes.

In [ ]:
clf = LogisticRegression().fit(Xtr, ytr)
pred = clf.predict(Xte)

### Step 3 — Read the confusion matrix and the metric report

The confusion matrix lays out the four outcomes — true negatives, false positives, false negatives, true positives — as `[[TN FP] [FN TP]]`. From those counts come **precision** (of the positives we predicted, how many were right), **recall** (of the real positives, how many we caught), and **F1** (their harmonic mean). The report prints these per class, exposing weaknesses on the rare class that overall accuracy would hide.

In [ ]:
print("confusion matrix [[TN FP] [FN TP]]:")
print(confusion_matrix(yte, pred))

print(classification_report(yte, pred, digits=3))

## Visualize the data & results

_On held-out breast-cancer tumors, how many did the model get right - and what kind of mistakes did it make?_

We train on real breast-cancer scans and draw the confusion matrix as a heatmap. We do it in two steps: (1) standardize the features, split, and fit, (2) predict and plot the confusion matrix so the mistakes are visible at a glance.

### Step 1 — Standardize, split, and fit

We load the real breast-cancer dataset and **standardize** every feature (subtract the mean, divide by the std) so no single large-valued column dominates the logistic fit. After splitting off a test set, we fit logistic regression with a higher iteration cap so it converges on the standardized features.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)   # standardize so no feature dominates
y = bc.target

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)

### Step 2 — Predict and plot the confusion matrix

We get predictions on the held-out test set and let scikit-learn draw the confusion matrix as a labeled heatmap. Rows are the true diagnosis, columns are the prediction; the bright diagonal is correct calls, and the off-diagonal cells show the two kinds of mistakes — and for a medical test, a false negative (missed malignant) is the costly one.

In [ ]:
pred = clf.predict(Xte)

# rows = true diagnosis, columns = predicted
ConfusionMatrixDisplay.from_predictions(
    yte, pred, display_labels=["malignant", "benign"])
plt.title("Confusion matrix (breast-cancer test set)")
plt.show()